In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, f1_score
import warnings
warnings.filterwarnings('ignore')


dfv1 = pd.read_csv('../data/processed_firecast_features_v3.csv')

X = dfv1.drop(['label', 'time'], axis=1)
y = dfv1['label']

# Temporal 90/10 split
split_idx = int(0.9 * len(dfv1))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Function Datasets
class WildfireDataset(Dataset):
    def __init__(self, features, labels):
        self.X = features
        self.y = labels.values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return torch.tensor(self.X[i], dtype=torch.float32), torch.tensor(self.y[i], dtype=torch.float32)

# CNN1D Model
class OptimizedCNN1D(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.3):
        super(OptimizedCNN1D, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2), nn.Dropout(dropout_rate)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2), nn.Dropout(dropout_rate)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4), nn.Dropout(dropout_rate)
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_dim)
            out = self.conv_block3(self.conv_block2(self.conv_block1(dummy)))
            self.flattened_size = out.view(1, -1).size(1)
        self.dense = nn.Sequential(
            nn.Linear(self.flattened_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.output = nn.Linear(128, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = x.view(x.size(0), -1)
        x = self.dense(x)
        return self.output(x).squeeze(1)


# --- Training Function ---
def train_model(model, train_loader, val_loader, epochs=150, lr=0.001):
    device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader)
    )
    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0
    patience, counter = 15, 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_logits.extend(logits.cpu().numpy())
                val_labels.extend(y.cpu().numpy())

        val_probs = torch.sigmoid(torch.tensor(val_logits)).numpy()
        if len(set(val_labels)) > 1:
            val_auc = roc_auc_score(val_labels, val_probs)
        else:
            val_auc = 0.0

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), "best_cnn1d_timeaware_baseline.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val AUC: {val_auc:.4f}")

    print(f"✅ Training finished. Best Val AUC: {best_auc:.4f}")
    return best_auc


# --- Train Model --- #
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = WildfireDataset(X_train_scaled, y_train)
val_dataset = WildfireDataset(X_test_scaled, y_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Training on device: {device}")

model = OptimizedCNN1D(input_dim=X_train_scaled.shape[1], dropout_rate=0.3).to(device)
train_model(model, train_loader, val_loader, epochs=150, lr=0.001)

# --- Evaluation --- #
model.load_state_dict(torch.load("best_cnn1d_timeaware_baseline.pth", map_location=device))
model.eval()

test_probs = []
with torch.no_grad():
    for x, _ in val_loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        test_probs.extend(probs)

test_probs = np.array(test_probs)
y_test_final = y_test.values

# --- Find Optimal Threshold ---
def find_threshold_for_recall(y_true, y_prob, target_recall=0.8):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    idx = np.where(recall >= target_recall)[0]
    if len(idx) == 0:
        return 0.5
    best_idx = idx[np.argmax(precision[idx])]
    return thresholds[best_idx]

# Option 1: Maximize F1
f1_scores = [f1_score(y_test_final, (test_probs > t).astype(int)) for t in np.arange(0.1, 0.9, 0.01)]
best_f1_thresh = np.arange(0.1, 0.9, 0.01)[np.argmax(f1_scores)]

# Option 2: Ensure 80% fire recall (recommended for wildfire)
recall_thresh = find_threshold_for_recall(y_test_final, test_probs, target_recall=0.80)

print(f"Best F1 threshold: {best_f1_thresh:.3f}")
print(f"80% fire recall threshold: {recall_thresh:.3f}")

# Use recall-focused threshold for final prediction
final_thresh = recall_thresh
y_pred = (test_probs > final_thresh).astype(int)

# --- Final Evaluation ---
print("\n" + "="*50)
print("FINAL MODEL EVALUATION (with tuned threshold)")
print("="*50)
print(f"AUC: {roc_auc_score(y_test_final, test_probs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_final, y_pred, target_names=['No Fire', 'Fire']))

Training on device: cuda
Epoch 1 | Loss: 0.6996 | Val AUC: 0.7107
Epoch 20 | Loss: 0.5520 | Val AUC: 0.8672
Epoch 40 | Loss: 0.5250 | Val AUC: 0.8729
Epoch 60 | Loss: 0.5063 | Val AUC: 0.8756
Early stopping at epoch 61
✅ Training finished. Best Val AUC: 0.8810
Best F1 threshold: 0.220
80% fire recall threshold: 0.369

FINAL MODEL EVALUATION (with tuned threshold)
AUC: 0.8810

Classification Report:
              precision    recall  f1-score   support

     No Fire       0.75      0.80      0.77      2447
        Fire       0.84      0.80      0.82      3175

    accuracy                           0.80      5622
   macro avg       0.79      0.80      0.80      5622
weighted avg       0.80      0.80      0.80      5622

